## imports

### reload

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### imports dependencies

In [28]:
import json
import sys
import os
import chromadb
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

# ingestion
from app.ingestion.chunker import chunk_docs
from app.ingestion.store import store_documents
from app.schemas import Document, DocumentEval
from app.rag.embeddings import MyEmbeddingFunctionSentenceTransformer

# rag 
from app.rag.pipeline import rag_pipeline

# dotenv 
from dotenv import load_dotenv

# visualization
from tqdm import tqdm


In [29]:
with open("../dataset/machine_learning_knowledge.json") as f:
    docs = json.load(f)
    print(docs[0])

{'id': 'doc1', 'text': 'Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'}


In [30]:
doc_source = [Document(id=doc["id"], content=doc["text"]) for doc in docs]

### setup storage

In [31]:
client = chromadb.PersistentClient(path="../chroma_db")
embedding_function = MyEmbeddingFunctionSentenceTransformer()
collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=embedding_function
)

Failed to send telemetry event ClientStartEvent: module 'chromadb' has no attribute '__version__'
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 759.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Failed to send telemetry event ClientCreateCollectionEvent: module 'chromadb' has no attribute '__version__'


### setup dotenv

In [32]:
load_dotenv(dotenv_path='../.env')

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## ingest

In [33]:
doc_source = chunk_docs(doc_source)
store_documents(collection, doc_source)

## rag pipeline

In [34]:
question = "What is machine learning and what is its main goal?"

In [35]:
answer, doc_retrieved = rag_pipeline(collection, question)
print(answer)
print()
for doc in doc_retrieved:
    print(doc.id)
    print(doc.content)
    print(doc.distance)
    print("=========================")

I do not know.

doc1
Machine learning is a field of artificial intelligence that focuses on enabling computers to learn p
0.6022921204566956


## evaluation

In [36]:
with open("../dataset/machine_learning_knowledge_evaluation.json") as f:
    doc_eval_json = json.load(f)
    print(doc_eval_json)

[{'question': 'What is machine learning and what is its main goal?', 'answer': 'Machine learning is a field of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without explicit programming.', 'relevant_doc_ids': ['doc1']}, {'question': 'Give two examples of applications where machine learning is commonly used.', 'answer': 'Machine learning is commonly used in applications such as recommendation systems, fraud detection, and image recognition.', 'relevant_doc_ids': ['doc1']}, {'question': 'What type of machine learning uses labeled data for training?', 'answer': 'Supervised learning uses labeled data where each training example includes input features and the correct output.', 'relevant_doc_ids': ['doc2']}, {'question': 'What are two common tasks in supervised learning?', 'answer': 'Two common supervised learning tasks are classification and regression.', 'relevant_doc_ids': ['doc2', 'doc8', 'doc9']}, {'question': 'What is the 

In [37]:
doc_eval = [DocumentEval(
                question=doc['question'], 
                answer=doc['answer'], 
                relevant_doc_ids=doc['relevant_doc_ids']) 
            for doc in doc_eval_json]

In [38]:
for doc in doc_eval:
    print(doc.question)
    print(doc.answer)
    print(doc.relevant_doc_ids)
    print("----------")

What is machine learning and what is its main goal?
Machine learning is a field of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without explicit programming.
['doc1']
----------
Give two examples of applications where machine learning is commonly used.
Machine learning is commonly used in applications such as recommendation systems, fraud detection, and image recognition.
['doc1']
----------
What type of machine learning uses labeled data for training?
Supervised learning uses labeled data where each training example includes input features and the correct output.
['doc2']
----------
What are two common tasks in supervised learning?
Two common supervised learning tasks are classification and regression.
['doc2', 'doc8', 'doc9']
----------
What is the main goal of unsupervised learning?
The goal of unsupervised learning is to discover hidden patterns or structures in unlabeled data.
['doc3']
----------
Name two techniques c

### run all cases

In [39]:
answers = []
retrieved_docs = []
for doc in tqdm(doc_eval):
    answer, retrieved_doc = rag_pipeline(collection, doc.question)
    if answer is None:
        answers.append("I don't know")
    else:
        answers.append(answer)

    retrieved_docs.append(retrieved_doc)
    

100%|██████████| 27/27 [02:55<00:00,  6.51s/it]


### recall@k

In [40]:
k = 5

In [41]:
total = 0.0
valid_cases = 0

def recall_at_k(relevant, retrieved, k):
    if not relevant:
        return None, set()   # undefined recall

    if not retrieved:
        return 0.0, set()

    retrieved_k = retrieved[:k]

    relevant_set = set(relevant)
    retrieved_set = set(retrieved_k)

    relevant_found = relevant_set.intersection(retrieved_set)

    recall = len(relevant_found) / len(relevant_set)

    return recall, relevant_found

for answer, retrieved, eval in tqdm(zip(answers, retrieved_docs, doc_eval)):
    relevant = eval.relevant_doc_ids

    print(f"answer\t\t: {answer}")
    print(f"relevant\t: {relevant}")

    # get retrieved
    if retrieved is None:
        score = 0.0
        print("retrieved\t: None")
    else:
        retrieved_ids = [r.id for r in retrieved]
        print(f"retrieved_ids\t: {retrieved_ids}")
        score, relevant_found = recall_at_k(relevant, retrieved_ids, k)
    
    print(f"recall@{k} score\t: {score}")
    print("=========================================")

    if score is not None:
        total += score
        valid_cases += 1


average = total / valid_cases if valid_cases > 0 else 0
print(f"average score: {average:.2f}")

27it [00:00, 5995.35it/s]

answer		: Machine Learning is a subfield of Artificial Intelligence (AI) that enables computers to automatically improve their performance on a task without being explicitly programmed. It's a type of intelligence that allows systems to learn from data, identify patterns, make predictions, or take actions based on that data.

The main goal of Machine Learning is to enable computers to:

* Learn from experience
* Improve their performance over time
* Adapt to new situations or data

Machine Learning involves training algorithms on large datasets, which allows them to learn the relationships between inputs and outputs. The goal is to create models that can make predictions, classify data, recognize patterns, or generate new content.

The specific goals of Machine Learning can vary depending on the context, but some common objectives include:

* Classification: Predicting categorical values (e.g., spam vs. non-spam emails)
* Regression: Predicting numerical values (e.g., stock prices)
* C

### generation evaluation - ragas

In [42]:
questions = [eval.question for eval in doc_eval]
ground_truths = [eval.answer for eval in doc_eval]

relevant_doc_ids   = [r.relevant_doc_ids for r in doc_eval]

contexts = []
for doc_list in retrieved_docs:
    if doc_list is None:
        contexts.append([])
    else:
        contexts.append([doc.content for doc in doc_list])

print(contexts)
print(ground_truths)


[['Machine learning is a field of artificial intelligence that focuses on enabling computers to learn p'], [], ['Supervised learning is a type of machine learning where models are trained using labeled data. Each '], [], ['Unsupervised learning involves training models on data that does not contain labels. The goal is to '], [], [], [], ['Overfitting occurs when a machine learning model learns the training data too well, including its no'], [], ['Underfitting happens when a model is too simple to capture the underlying structure of the data. In ', 'Overfitting occurs when a machine learning model learns the training data too well, including its no'], ['Feature engineering is the process of selecting, transforming, or creating variables that improve mo'], ['Regression is a supervised learning task where the goal is to predict a continuous numerical value. '], [], ['Classification is a machine learning task where the goal is to assign inputs into predefined categor', 'Supervised learning

In [43]:
for i in range(len(answers)):
    if answers[i] is None or answers[i] == "":
        print("Empty answer at index", i)

In [44]:
from datasets import Dataset

print(f"question len    : {len(questions)}")
print(f"contexts len    : {len(contexts)}")
print(f"response len    : {len(answers)}")
print(f"ground_truth len: {len(ground_truths)}")

data = {
    "question": questions,
    "contexts": contexts,
    "answer": answers,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(data)

question len    : 27
contexts len    : 27
response len    : 27
ground_truth len: 27


In [45]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from app.config import EVALUATION_MODEL

"""
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0,
    model_kwargs={"n": 1}
)
"""

llm = ChatOpenAI(
    model=EVALUATION_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    model_kwargs={"n": 3}
)

ragas_llm = LangchainLLMWrapper(llm)

d:\Programs\anaconda3\envs\rag_py3.10\lib\site-packages\IPython\core\interactiveshell.py:3519: UserWarning: Parameters {'n'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):
C:\Users\april\AppData\Local\Temp\ipykernel_13332\2438820990.py:22: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)


In [46]:
from langchain_core.embeddings import Embeddings

class RagasEmbeddingWrapper(Embeddings):
    def __init__(self, embedding_function):
        self.embedding_function = embedding_function

    def embed_documents(self, texts):
        return self.embedding_function(texts)

    def embed_query(self, text):
        return self.embedding_function([text])[0]

embeddings = RagasEmbeddingWrapper(embedding_function)

In [47]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=embeddings
)

C:\Users\april\AppData\Local\Temp\ipykernel_13332\1542496564.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_13332\1542496564.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_13332\1542496564.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_13

In [48]:
print(result)

{'faithfulness': 0.1372, 'answer_relevancy': 0.3025, 'context_precision': 0.5185, 'context_recall': 0.5556}


In [49]:
df = result.to_pandas()
df.head()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is machine learning and what is its main ...,[Machine learning is a field of artificial int...,Machine Learning is a subfield of Artificial I...,Machine learning is a field of artificial inte...,0.344828,0.984732,0.0,0.0
1,Give two examples of applications where machin...,[],I don't know,Machine learning is commonly used in applicati...,0.000000,0.000000,0.0,0.0
2,What type of machine learning uses labeled dat...,[Supervised learning is a type of machine lear...,Supervised learning is a type of machine learn...,Supervised learning uses labeled data where ea...,0.500000,0.732788,1.0,1.0
3,What are two common tasks in supervised learning?,[],I don't know,Two common supervised learning tasks are class...,0.000000,0.000000,0.0,0.0
4,What is the main goal of unsupervised learning?,[Unsupervised learning involves training model...,I don't know.,The goal of unsupervised learning is to discov...,0.000000,0.000000,1.0,1.0
